In [11]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transform
import torch.nn.functional as F

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset=torchvision.datasets.MNIST(root='./data',train=True,transform=transform.ToTensor(),download=True)
train_loader=torch.utils.data.DataLoader(dataset=train_dataset,batch_size=512)
test_dataset=torchvision.datasets.MNIST(root='./data',train=False,transform=transform.ToTensor())
test_loader=torch.utils.data.DataLoader(dataset=test_dataset,batch_size=512)

In [12]:
class ConvNeuralNet(nn.Module):
    def __init__(self):
        super(ConvNeuralNet,self).__init__()
        self.conv1=nn.Conv2d(1,16,3)
        self.pool=nn.MaxPool2d(2,2)
        self.conv2=nn.Conv2d(16,32,3)
        self.fc1=nn.Linear(32*5*5,100)
        self.fc2=nn.Linear(100,64)
        self.fc3=nn.Linear(64,10)

    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        x=x.view(-1,32*5*5)
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=self.fc3(x)
        return x

model=ConvNeuralNet().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.01)

n_class_correct=[0 for _ in range(10)]
n_class_samples=[0 for _ in range(10)]

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(device)
print(next(model.parameters()).device)

for epoch in range(50):
    for images,labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)
        output=model(images)
        loss=criterion(output,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')

with torch.no_grad():
    n_correct=0
    for images,labels in test_loader:
        images=images.to(device)
        labels=labels.to(device)
        output=model(images)
        _,pred=torch.max(output,1)
        n_correct+=(pred==labels).sum().item()
        for i in range(len(labels)):
            label=labels[i].item()
            prediction=pred[i].item()
            if(label==prediction):
                n_class_correct[label]+=1
            n_class_samples[label]+=1
    print(f"Overall Accuracy:{n_correct/len(test_loader.dataset)}")
    for i in range(10):
        print(f"Accuracy of class {i}:{n_class_correct[i]/n_class_samples[i]}")

2.11.0+cu128
12.8
True
cuda
cuda:0
Epoch 10, Loss: 1.1732
Epoch 20, Loss: 0.3918
Epoch 30, Loss: 0.3156
Epoch 40, Loss: 0.2721
Epoch 50, Loss: 0.2545
Overall Accuracy:0.9637
Accuracy of class 0:0.9887755102040816
Accuracy of class 1:0.9770925110132158
Accuracy of class 2:0.9447674418604651
Accuracy of class 3:0.9594059405940594
Accuracy of class 4:0.9755600814663951
Accuracy of class 5:0.9820627802690582
Accuracy of class 6:0.9770354906054279
Accuracy of class 7:0.9095330739299611
Accuracy of class 8:0.9640657084188912
Accuracy of class 9:0.9623389494549058
